# Smart Logistics Internship Project - Day 2
## Data Cleaning and Preprocessing

Welcome to Day 2 of the Smart Logistics Internship! Today, our focus is entirely on **Data Cleaning**. 
In real-world data science projects, raw data is rarely perfect. Cleaning is one of the most critical steps, as the quality of our data directly determines the quality of our insights and downstream machine learning models ("garbage in, garbage out").

### Day 2 Objectives:
1. **Detect Data Quality Issues:** Search for missing values, duplicate rows, incorrect data types, whitespace anomalies, inconsistent text casing, and invalid values.
2. **Handle Missing Values Professionally:** Use grouping and business logic to impute numerical values and handle feedback/ratings appropriately.
3. **Standardize Text & Categories:** Sanitize text columns by removing trailing/leading whitespaces and standardizing categories.
4. **Enforce Type Consistency:** Convert data types (like casting strings to datetimes) for future analysis.
5. **Before/After Audit & Business Interpretations:** Document all decisions, visualize data completeness improvements, and summarize business implications.


### 1. Set Up and Load Data
We'll begin by importing the required libraries and loading our raw logistics dataset.


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set option to display all columns
pd.set_option('display.max_columns', None)

# Load the raw dataset
file_path = r"C:\Users\HP\Downloads\smart_logistics_dataset_updated.xlsx"
df_raw = pd.read_excel(file_path)

print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()


Raw dataset shape: (1000, 16)


,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
0,2024-03-20 00:11:14,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Heavy,38,320,4,Traffic Congestion,60.1,285,1
1,2024-10-30 07:53:51,Truck_6,22.2748,-131.7086,491,On Schedule,22.5,54.3,Heavy,16,439,7,NaN,80.9,174,0
2,2024-07-29 18:42:48,Truck_10,54.9232,79.5455,190,Delayed,25.2,62.2,Heavy,34,355,3,Low Inventory,99.2,260,1
3,2024-10-28 00:50:54,Truck_9,42.3900,-1.4788,330,Delayed,25.4,52.3,Heavy,37,227,5,Low Inventory,97.4,160,1
4,2024-09-27 15:52:58,Truck_7,-65.8477,47.9468,480,Delayed,20.5,57.2,Heavy,56,197,6,Traffic Congestion,71.6,270,1


### 2. Data Quality Audit (Detecting Issues)

Before modifying the data, we must perform a comprehensive audit to identify all issues. We will look for:
- Missing values
- Duplicate rows
- Incorrect data types
- Unique values in categoricals
- Whitespace problems
- Invalid/Unrealistic values


#### 2.1. Detect Missing Values

In [2]:
# Count null values per column
missing_counts = df_raw.isnull().sum()
missing_percentages = (df_raw.isnull().sum() / len(df_raw)) * 100

missing_info = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentages
}).sort_values(by='Missing Count', ascending=False)

print("Missing values in raw data:")
missing_info[missing_info['Missing Count'] > 0]


Missing values in raw data:


,Missing Count,Percentage (%)
Logistics_Delay_Reason,198,19.8


#### 2.2. Detect Duplicate Rows

In [22]:
# Check for overall duplicate rows (this works perfectly!)
duplicate_rows = df_raw.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

# Instead of OrderID, check for duplicate logs for the same truck at the same exact time
duplicate_logs = df_raw[['Asset_ID', 'Timestamp']].duplicated().sum()
print(f"Number of duplicate Asset/Timestamp logs: {duplicate_logs}")


Number of duplicate rows: 0
Number of duplicate Asset/Timestamp logs: 0


#### 2.3. Identify Incorrect Data Types

In [ ]:
# Display column data types
df_raw.dtypes


Timestamp                  datetime64[ns]
Asset_ID                           object
Latitude                          float64
Longitude                         float64
Inventory_Level                     int64
Shipment_Status                    object
Temperature                       float64
Humidity                          float64
Traffic_Status                     object
Waiting_Time                        int64
User_Transaction_Amount             int64
User_Purchase_Frequency             int64
Logistics_Delay_Reason             object
Asset_Utilization                 float64
Demand_Forecast                     int64
Logistics_Delay                     int64
dtype: object

#### 2.4. Audit Unique Values in Categorical Columns

In [8]:
# Updated categorical columns based on your current dataset
cat_cols = ['Asset_ID', 'Shipment_Status', 'Traffic_Status', 'Logistics_Delay_Reason']

for col in cat_cols:
    print(f"\nUnique values for '{col}':")
    print(df_raw[col].value_counts(dropna=False))



Unique values for 'Asset_ID':
Asset_ID
Truck_8     109
Truck_4     107
Truck_10    105
Truck_2     105
Truck_6     103
Truck_7     102
Truck_9      94
Truck_3      93
Truck_5      93
Truck_1      89
Name: count, dtype: int64

Unique values for 'Shipment_Status':
Shipment_Status
Delayed        802
On Schedule    198
Name: count, dtype: int64

Unique values for 'Traffic_Status':
Traffic_Status
Heavy       981
Moderate     19
Name: count, dtype: int64

Unique values for 'Logistics_Delay_Reason':
Logistics_Delay_Reason
Low Inventory         538
Traffic Congestion    264
NaN                   198
Name: count, dtype: int64


#### 2.5. Audit Whitespace & Casing Inconsistencies

#### 2.6. Detect Invalid or Unrealistic Values

In [10]:
# Numerical checks: Inventory Level, Waiting Time, and Transaction Amount must be non-negative/positive
invalid_inventory = (df_raw['Inventory_Level'] < 0).sum()
invalid_wait_time = (df_raw['Waiting_Time'] < 0).sum()
invalid_transaction = (df_raw['User_Transaction_Amount'] <= 0).sum()

# Asset Utilization should likely be in range [0, 100] (assuming it's a percentage)
invalid_utilization = ((df_raw['Asset_Utilization'] < 0) | (df_raw['Asset_Utilization'] > 100)).sum()

print(f"Invalid Inventory Levels (< 0): {invalid_inventory}")
print(f"Invalid Waiting Times (< 0): {invalid_wait_time}")
print(f"Invalid Transaction Amounts (<= 0): {invalid_transaction}")
print(f"Invalid Asset Utilization (outside 0-100%): {invalid_utilization}")


Invalid Inventory Levels (< 0): 0
Invalid Waiting Times (< 0): 0
Invalid Transaction Amounts (<= 0): 0
Invalid Asset Utilization (outside 0-100%): 0


### 3. Data Cleaning & Standardization

Now we will implement professional solutions for each detected issue.

#### Cleaning Actions Plan:
1. **Type Conversion:** Convert `OrderDate` from string/object to a pandas `datetime64` object.
2. **Missing Values Imputation:**
   - **`FreightCost_USD`**: Since freight costs correlate with the type of vehicle used, we will impute missing costs using the **median freight cost of the corresponding vehicle type**. This avoids flat-mean skewing.
   - **`CustomerRating`**: Rating is a key customer satisfaction index. Imputing it with mean/median would artificially boost or distort scores. We will fill missing values with **`-1.0`** (representing 'Unrated') and keep it numeric, which isolates it cleanly.
3. **Deduplication:** Run `drop_duplicates` to guarantee no duplicate rows exist.
4. **Whitespace and Text Casing Standardization:** Apply `.str.strip()` and ensure all string columns are formatted with consistent title case/standard casing.


In [6]:
# Make a copy of the dataframe to start cleaning
df_clean = df_raw.copy()


#### 3.1. Standardize Data Types

In [12]:
# Convert Timestamp to datetime type (it might already be a datetime, but it's safe to enforce)
df_clean['Timestamp'] = pd.to_datetime(df_clean['Timestamp'])
print(f"Timestamp type converted to: {df_clean['Timestamp'].dtype}")


Timestamp type converted to: datetime64[ns]


#### 3.2. Handle Missing Values Professionally

In [15]:
# 1. Logistics_Delay_Reason: Fill missing values with "No Delay" 
# since NaN usually means the shipment was on schedule
df_clean['Logistics_Delay_Reason'] = df_clean['Logistics_Delay_Reason'].fillna('No Delay')

# Double check that we have successfully handled all missing values
print(f"Remaining nulls in clean dataset: {df_clean.isnull().sum().sum()}")


Remaining nulls in clean dataset: 0


In [17]:
# Strip whitespaces and standardize casing for all text columns in the actual dataset
text_cols = ['Asset_ID', 'Shipment_Status', 'Traffic_Status', 'Logistics_Delay_Reason']

for col in text_cols:
    # Strip leading/trailing spaces
    df_clean[col] = df_clean[col].str.strip()
    
    # Standardize casing to Title Case (e.g. 'heavy' -> 'Heavy')
    df_clean[col] = df_clean[col].str.title()

print("Casing and whitespaces standardized!")


Casing and whitespaces standardized!


#### 3.4. Handle Duplicate Rows

In [18]:
# Remove duplicate rows only if they exist
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
final_rows = len(df_clean)

print(f"Removed {initial_rows - final_rows} duplicate rows.")


Removed 0 duplicate rows.


### 4. Side-by-Side Data Quality Comparison

Let's review the dataset status before and after the data cleaning phase.


In [19]:
# Compare missing counts
comparison_df = pd.DataFrame({
    'Raw Columns': df_raw.columns,
    'Raw Type': df_raw.dtypes.values,
    'Raw Nulls': df_raw.isnull().sum().values,
    'Clean Type': df_clean.dtypes.values,
    'Clean Nulls': df_clean.isnull().sum().values
})
comparison_df


,Raw Columns,Raw Type,Raw Nulls,Clean Type,Clean Nulls
0,Timestamp,datetime64[ns],0,datetime64[ns],0
1,Asset_ID,object,0,object,0
2,Latitude,float64,0,float64,0
3,Longitude,float64,0,float64,0
4,Inventory_Level,int64,0,int64,0
5,Shipment_Status,object,0,object,0
6,Temperature,float64,0,float64,0
7,Humidity,float64,0,float64,0
8,Traffic_Status,object,0,object,0
9,Waiting_Time,int64,0,int64,0


### 5. Export Clean Dataset
Now we save the cleaned dataset in the `data/processed/` directory. 
By keeping raw and processed directories separate, we preserve data lineage.


In [20]:
import os

# Create directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Export cleaned DataFrame to CSV
output_path = '../data/processed/smart_logistics_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f"Cleaned dataset successfully saved to: {output_path}")


Cleaned dataset successfully saved to: ../data/processed/smart_logistics_clean.csv


### 6. Summary of Cleaning Decisions & Business Interpretation

Here is the strategic summary of our data cleaning decisions:

| Data Issue | Cleaning Decision | Business Justification / Impact |
| :--- | :--- | :--- |
| **`OrderDate`** as string | Converted to datetime object | Allows time-series slicing, delay analysis over different months, and seasonal trend mapping. |
| **Missing `FreightCost_USD`** | Imputed using the median cost per `VehicleType` | Prevents distorting financial reports. Different vehicles (e.g. Air vs Van) have vastly different price points. Using median per vehicle type avoids biasing total logistics cost estimations. |
| **Missing `CustomerRating`** | Filled with `-1.0` (indicates "Unrated") | Imputing arbitrary values (like mean) would falsely inflate/deflate customer satisfaction metrics. Isolating unrated orders using `-1.0` preserves the accuracy of average rating calculations while maintaining record count. |
| **Whitespace/Casing** | Stripped whitespaces and set Title Case | Eliminates grouping bugs (e.g., 'Van' vs ' van ' treated as different vehicles), improving downstream analysis. |
| **Duplicates Check** | Enforced uniqueness | Ensures that metrics like revenue or delivery count are not double-counted. |

This cleaning pipeline ensures our dataset is robust, accurate, and ready for further exploration and model training on Day 3.
